# 04 · Clustering con PSO
Partícula = los K centros pegados. Aptitud = inercia.
K-Means es el rival especialista: suele ganar. Se dice de frente.


In [ ]:
# Motor ABC + PSO (incluido para que Colab no dependa de rutas)
#!/usr/bin/env python3
"""Primitivas ABC y PSO usadas por los cuatro ejercicios."""

import numpy as np


def pso_minimize(objective, bounds, n_particles=20, iters=25, w=0.7, c1=1.5, c2=1.5, seed=42):
    """PSO global-best. Minimiza objective(x).

    Ciclo: representacion = posicion continua,
    inicializacion uniforme, aptitud = objective,
    comportamiento = inercia + pbest + gbest,
    evolucion por iteraciones, parada = iters.
    """
    rng = np.random.default_rng(seed)
    lo, hi = np.asarray(bounds[0], float), np.asarray(bounds[1], float)
    dim = lo.size
    pos = rng.uniform(lo, hi, size=(n_particles, dim))
    vel = np.zeros_like(pos)
    costs = np.array([objective(p) for p in pos])
    pbest, pbest_c = pos.copy(), costs.copy()
    g = int(np.argmin(pbest_c))
    gbest, gbest_c = pbest[g].copy(), float(pbest_c[g])
    hist = [gbest_c]
    swarm_hist = [pos.copy()]
    for _ in range(iters):
        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        vel = w * vel + c1 * r1 * (pbest - pos) + c2 * r2 * (gbest - pos)
        pos = np.clip(pos + vel, lo, hi)
        costs = np.array([objective(p) for p in pos])
        improved = costs < pbest_c
        pbest[improved] = pos[improved]
        pbest_c[improved] = costs[improved]
        g = int(np.argmin(pbest_c))
        if pbest_c[g] < gbest_c:
            gbest, gbest_c = pbest[g].copy(), float(pbest_c[g])
        hist.append(gbest_c)
        swarm_hist.append(pos.copy())
    return gbest, gbest_c, np.array(hist), swarm_hist


def abc_binary_maximize(objective, n_bits, n_bees=10, cycles=12, limit=4, seed=42):
    """ABC binario. Maximiza objective(bitstring).

    Ciclo: representacion = fuente de alimento binaria,
    inicializacion aleatoria, aptitud = objective,
    comportamiento = empleada / observadora / exploradora,
    evolucion por ciclos, parada = cycles.
    """
    rng = np.random.default_rng(seed)
    foods = rng.integers(0, 2, size=(n_bees, n_bits))
    empty = foods.sum(axis=1) == 0
    if empty.any():
        foods[empty, rng.integers(0, n_bits, size=int(empty.sum()))] = 1
    fit = np.array([objective(f) for f in foods])
    trials = np.zeros(n_bees, dtype=int)
    best_i = int(np.argmax(fit))
    best, best_f = foods[best_i].copy(), float(fit[best_i])
    hist = [best_f]

    def neighbor(src):
        k = int(rng.integers(0, n_bits))
        nxt = src.copy()
        nxt[k] ^= 1
        if nxt.sum() == 0:
            nxt[k] = 1
        return nxt

    for _ in range(cycles):
        for i in range(n_bees):
            cand = neighbor(foods[i])
            fc = objective(cand)
            if fc >= fit[i]:
                foods[i], fit[i], trials[i] = cand, fc, 0
            else:
                trials[i] += 1
        probs = fit - fit.min() + 1e-9
        probs = probs / probs.sum()
        for _o in range(n_bees):
            i = int(rng.choice(n_bees, p=probs))
            cand = neighbor(foods[i])
            fc = objective(cand)
            if fc >= fit[i]:
                foods[i], fit[i], trials[i] = cand, fc, 0
            else:
                trials[i] += 1
        for i in range(n_bees):
            if trials[i] >= limit:
                foods[i] = rng.integers(0, 2, size=n_bits)
                if foods[i].sum() == 0:
                    foods[i][int(rng.integers(0, n_bits))] = 1
                fit[i] = objective(foods[i])
                trials[i] = 0
        bi = int(np.argmax(fit))
        if fit[bi] > best_f:
            best, best_f = foods[bi].copy(), float(fit[bi])
        hist.append(best_f)
    return best, best_f, np.array(hist)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine, make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

def assign(X, C):
    d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    return d.argmin(1), float(d.min(axis=1).sum())

X, _ = make_blobs(n_samples=360, centers=4, cluster_std=1.15, random_state=SEED)
K = 4

def inertia(pos):
    _, J = assign(X, pos.reshape(K, 2))
    return J

bounds = (np.repeat(X.min(0), K), np.repeat(X.max(0), K))
best, cost, hist, _ = pso_minimize(inertia, bounds, n_particles=18, iters=22, seed=SEED)
C_pso = best.reshape(K, 2)
lab_pso, _ = assign(X, C_pso)
sil_pso = float(silhouette_score(X, lab_pso))
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X)
sil_km = float(silhouette_score(X, km.labels_))
print(f"inercia PSO={cost:.1f}  KM={km.inertia_:.1f}")
print(f"silueta PSO={sil_pso:.3f}  KM={sil_km:.3f}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.6, 4), sharex=True, sharey=True)
ax[0].scatter(X[:,0], X[:,1], c=lab_pso, cmap="tab10", s=14)
ax[0].scatter(C_pso[:,0], C_pso[:,1], c="k", marker="X", s=90)
ax[0].set_title(f"PSO  sil={sil_pso:.3f}")
ax[1].scatter(X[:,0], X[:,1], c=km.labels_, cmap="tab10", s=14)
ax[1].scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1], c="k", marker="X", s=90)
ax[1].set_title(f"K-Means  sil={sil_km:.3f}")
plt.tight_layout(); plt.show()


In [ ]:
# Wine, solo 2 columnas (para dibujar)
wine = load_wine()
Xw = StandardScaler().fit_transform(wine.data[:, :2])
Kw = 3
def inertia_w(pos):
    _, J = assign(Xw, pos.reshape(Kw, 2))
    return J
bw = (np.repeat(Xw.min(0), Kw), np.repeat(Xw.max(0), Kw))
bestw, _, _, _ = pso_minimize(inertia_w, bw, n_particles=14, iters=18, seed=3)
Cw = bestw.reshape(Kw, 2)
labw, _ = assign(Xw, Cw)
kmw = KMeans(n_clusters=Kw, n_init=10, random_state=SEED).fit(Xw)
print("Wine2D silueta PSO", float(silhouette_score(Xw, labw)), "KM", float(silhouette_score(Xw, kmw.labels_)))
fig, ax = plt.subplots(figsize=(5.2, 4.2))
ax.scatter(Xw[:,0], Xw[:,1], c=labw, cmap="tab10", s=18)
ax.scatter(Cw[:,0], Cw[:,1], c="k", marker="X", s=90)
ax.set_title("Wine 2 variables — centros PSO")
plt.tight_layout(); plt.show()
